In [5]:
import struct
import socket

In [6]:
def build_dns_query(domain_name, query_type=1):
    """
    Constrói uma mensagem de consulta DNS (A record) conforme RFC 1035.
    """
    # 1. Cabeçalho (Header)
    # ID: Random, Flags: 0x0100 (Recursion Desired), QDCOUNT: 1, ANCOUNT: 0, NSCOUNT: 0, ARCOUNT: 0
    transaction_id = 0xAAAA
    flags = 0x0100
    qdcount = 1
    ancount = 0
    nscount = 0
    arcount = 0
    
    # '!' = network order (big-endian), H = unsigned short (2 bytes)
    header = struct.pack('!HHHHHH', transaction_id, flags, qdcount, ancount, nscount, arcount)

    # 2. Pergunta (Question) - Nome codificado (ex: www.google.com -> \x03www\x06google\x03com\x00)
    encoded_name = b""
    for part in domain_name.encode("ascii").split(b"."):
        encoded_name += bytes([len(part)]) + part
    encoded_name += b"\x00"  # Finalizador

    # Tipo: A (1) | Classe: IN (1)
    qtype = query_type
    qclass = 1 # IN
    
    question = encoded_name + struct.pack('!HH', qtype, qclass)

    return header + question

domain = "google.com"
dns_message = build_dns_query(domain)

print(f"Mensagem DNS para {domain} ({len(dns_message)} bytes):")
print(dns_message)

Mensagem DNS para google.com (28 bytes):
b'\xaa\xaa\x01\x00\x00\x01\x00\x00\x00\x00\x00\x00\x06google\x03com\x00\x00\x01\x00\x01'


In [12]:
dns_query = dns_message

DNS_SERVER = "8.8.8.8"
PORT = 53

# 3. Criar socket UDP (SOCK_DGRAM)
# Note: Raw sockets (SOCK_RAW) costumam exigir privilégios de root/admin,
# mas SOCK_DGRAM envia o payload DNS "cru" dentro de um UDP gerado pelo SO.
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

try:
    print(f"Enviando query para {DNS_SERVER}...")
    
    # 4. Enviar a mensagem pronta
    sock.sendto(dns_query, (DNS_SERVER, PORT))
    
    # 5. Receber resposta
    data, addr = sock.recvfrom(1024)

finally:
    sock.close()

print(f"Resposta recebida de {addr}: ")
print(data)

Enviando query para 8.8.8.8...
Resposta recebida de ('8.8.8.8', 53): 
b'\xaa\xaa\x81\x80\x00\x01\x00\x01\x00\x00\x00\x00\x06google\x03com\x00\x00\x01\x00\x01\xc0\x0c\x00\x01\x00\x01\x00\x00\x01&\x00\x04\xac\xd9\x1d\xce'


In [14]:
DNS_TYPE_NAMES = {
    1: "A",
    2: "NS",
    5: "CNAME",
    6: "SOA",
    12: "PTR",
    15: "MX",
    16: "TXT",
    28: "AAAA",
}

def decode_dns_name(packet, offset):
    labels = []
    jumped = False
    next_offset = offset

    while True:
        length = packet[offset]

        if length & 0xC0 == 0xC0:
            pointer = ((length & 0x3F) << 8) | packet[offset + 1]
            if not jumped:
                next_offset = offset + 2
            offset = pointer
            jumped = True
            continue

        if length == 0:
            offset += 1
            if not jumped:
                next_offset = offset
            break

        offset += 1
        labels.append(packet[offset : offset + length].decode("ascii"))
        offset += length

    return ".".join(labels), next_offset

def decode_dns_response(packet):
    transaction_id, flags, qdcount, ancount, nscount, arcount = struct.unpack_from('!HHHHHH', packet, 0)
    offset = 12

    qr = bool(flags & 0x8000)
    aa = bool(flags & 0x0400)
    tc = bool(flags & 0x0200)
    rd = bool(flags & 0x0100)
    ra = bool(flags & 0x0080)
    rcode = flags & 0x000F

    lines = [
        f"ID da transação: 0x{transaction_id:04x}",
        f"Flags: 0x{flags:04x}",
        f"  - resposta: {qr}",
        f"  - autoritária: {aa}",
        f"  - truncada: {tc}",
        f"  - recursion desired: {rd}",
        f"  - recursion available: {ra}",
        f"  - rcode: {rcode}",
        f"Perguntas: {qdcount}",
        f"Respostas: {ancount}",
        f"Autoritativas: {nscount}",
        f"Adicionais: {arcount}",
        "",
        "Pergunta:",
    ]

    for index in range(qdcount):
        name, offset = decode_dns_name(packet, offset)
        qtype, qclass = struct.unpack_from('!HH', packet, offset)
        offset += 4
        lines.append(f"  {index + 1}. nome={name}, tipo={DNS_TYPE_NAMES.get(qtype, qtype)}, classe={qclass}")

    lines.extend(["", "Respostas:"])

    for index in range(ancount):
        name, offset = decode_dns_name(packet, offset)
        rtype, rclass, ttl, rdlength = struct.unpack_from('!HHIH', packet, offset)
        offset += 10
        rdata = packet[offset : offset + rdlength]
        offset += rdlength

        if rtype == 1 and rdlength == 4:
            rdata_text = socket.inet_ntoa(rdata)
        else:
            rdata_text = rdata.hex()

        lines.append(f"  {index + 1}. nome={name}, tipo={DNS_TYPE_NAMES.get(rtype, rtype)}, ttl={ttl}s, dado={rdata_text}")

    return "\n".join(lines)

print(f"Resposta decodificada de {addr}:")
print(decode_dns_response(data))

Resposta decodificada de ('8.8.8.8', 53):
ID da transação: 0xaaaa
Flags: 0x8180
  - resposta: True
  - autoritária: False
  - truncada: False
  - recursion desired: True
  - recursion available: True
  - rcode: 0
Perguntas: 1
Respostas: 1
Autoritativas: 0
Adicionais: 0

Pergunta:
  1. nome=google.com, tipo=A, classe=1

Respostas:
  1. nome=google.com, tipo=A, ttl=294s, dado=172.217.29.206


In [18]:
import time
import random
from collections import Counter
from statistics import mean

DNS_SERVERS = [
    "8.8.8.8",
    "1.1.1.1",
    "9.9.9.9",
    "208.67.222.222",
]

DNS_RCODE_NAMES = {
    0: "NOERROR",
    1: "FORMERR",
    2: "SERVFAIL",
    3: "NXDOMAIN",
    4: "NOTIMP",
    5: "REFUSED",
}

def build_dns_query(domain_name, query_type=1, transaction_id=None):
    if transaction_id is None:
        transaction_id = random.randint(0, 0xFFFF)

    flags = 0x0100
    header = struct.pack("!HHHHHH", transaction_id, flags, 1, 0, 0, 0)

    encoded_name = b""
    for part in domain_name.encode("ascii").split(b"."):
        encoded_name += bytes([len(part)]) + part
    encoded_name += b"\x00"

    question = encoded_name + struct.pack("!HH", query_type, 1)
    return header + question

def decode_dns_name(packet, offset):
    labels = []
    jumped = False
    next_offset = offset

    while True:
        length = packet[offset]

        if length & 0xC0 == 0xC0:
            pointer = ((length & 0x3F) << 8) | packet[offset + 1]
            if not jumped:
                next_offset = offset + 2
            offset = pointer
            jumped = True
            continue

        if length == 0:
            offset += 1
            if not jumped:
                next_offset = offset
            break

        offset += 1
        labels.append(packet[offset : offset + length].decode("ascii"))
        offset += length

    return ".".join(labels), next_offset

def decode_dns_response(packet):
    transaction_id, flags, qdcount, ancount, nscount, arcount = struct.unpack_from("!HHHHHH", packet, 0)
    offset = 12

    qr = bool(flags & 0x8000)
    aa = bool(flags & 0x0400)
    tc = bool(flags & 0x0200)
    rd = bool(flags & 0x0100)
    ra = bool(flags & 0x0080)
    rcode = flags & 0x000F

    questions = []
    answers = []

    for _ in range(qdcount):
        name, offset = decode_dns_name(packet, offset)
        qtype, qclass = struct.unpack_from("!HH", packet, offset)
        offset += 4
        questions.append({"name": name, "type": DNS_TYPE_NAMES.get(qtype, qtype), "class": qclass})

    for _ in range(ancount):
        name, offset = decode_dns_name(packet, offset)
        rtype, rclass, ttl, rdlength = struct.unpack_from("!HHIH", packet, offset)
        offset += 10
        rdata = packet[offset : offset + rdlength]
        offset += rdlength

        if rtype == 1 and rdlength == 4:
            rdata_text = socket.inet_ntoa(rdata)
        elif rtype == 28 and rdlength == 16:
            rdata_text = socket.inet_ntop(socket.AF_INET6, rdata)
        else:
            rdata_text = rdata.hex()

        answers.append({
            "name": name,
            "type": DNS_TYPE_NAMES.get(rtype, rtype),
            "rtype": rtype,
            "ttl": ttl,
            "data": rdata_text,
        })

    return {
        "transaction_id": transaction_id,
        "flags": flags,
        "rcode": rcode,
        "rcode_name": DNS_RCODE_NAMES.get(rcode, f"RCODE_{rcode}"),
        "qr": qr,
        "aa": aa,
        "tc": tc,
        "rd": rd,
        "ra": ra,
        "qdcount": qdcount,
        "ancount": ancount,
        "nscount": nscount,
        "arcount": arcount,
        "questions": questions,
        "answers": answers,
    }

def query_dns_server(server, domain_name, timeout=2.0, query_type=1):
    start = time.perf_counter()
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.settimeout(timeout)

    try:
        query = build_dns_query(domain_name, query_type=query_type)
        sock.sendto(query, (server, 53))
        data, _ = sock.recvfrom(2048)
        elapsed = time.perf_counter() - start
        decoded = decode_dns_response(data)

        ip_answers = [answer["data"] for answer in decoded["answers"] if answer["rtype"] == 1]
        return {
            "server": server,
            "status": "ok",
            "elapsed": elapsed,
            "decoded": decoded,
            "ips": ip_answers,
            "error": None,
        }
    except socket.timeout:
        elapsed = time.perf_counter() - start
        return {
            "server": server,
            "status": "timeout",
            "elapsed": elapsed,
            "decoded": None,
            "ips": [],
            "error": "timeout",
        }
    except OSError as exc:
        elapsed = time.perf_counter() - start
        return {
            "server": server,
            "status": "error",
            "elapsed": elapsed,
            "decoded": None,
            "ips": [],
            "error": str(exc),
        }
    finally:
        sock.close()

def run_multi_server_scan(domain_name, servers=DNS_SERVERS, timeout=2.0):
    results = []
    for server in servers:
        result = query_dns_server(server, domain_name, timeout=timeout)
        results.append(result)

    consensus = Counter()
    for result in results:
        for ip in result["ips"]:
            consensus[ip] += 1
    consensus_ip = consensus.most_common(1)[0][0] if consensus else None

    rows = []
    for result in results:
        decoded = result["decoded"]
        rcode_name = decoded["rcode_name"] if decoded else "N/A"
        ips = result["ips"]
        rows.append({
            "server": result["server"],
            "status": result["status"],
            "time_ms": round(result["elapsed"] * 1000, 2),
            "rcode": rcode_name,
            "ips": ips,
            "consensus_match": bool(consensus_ip and consensus_ip in ips),
            "notes": [],
        })

    majority_resolution = None
    if consensus_ip:
        majority_resolution = consensus_ip

    for row in rows:
        if row["status"] == "timeout":
            row["notes"].append("timeout")
            continue
        if row["rcode"] == "NXDOMAIN" and majority_resolution:
            row["notes"].append("possible_blocking_nxdomain")
        if row["rcode"] == "REFUSED":
            row["notes"].append("refused")
        if any(ip in {"0.0.0.0", "127.0.0.1"} for ip in row["ips"]):
            row["notes"].append("nulled_ip")
        if majority_resolution and row["ips"] and majority_resolution not in row["ips"]:
            row["notes"].append("ip_divergent")
        if row["rcode"] == "NOERROR" and not row["ips"]:
            row["notes"].append("no_a_records")

    return results, rows, majority_resolution


In [19]:
def benchmark_servers(domain_name, servers=DNS_SERVERS, iterations=10, timeout=2.0):
    stats = []

    for server in servers:
        samples = []
        failures = 0
        rcodes = []

        for _ in range(iterations):
            result = query_dns_server(server, domain_name, timeout=timeout)
            if result["status"] == "ok" and result["decoded"] is not None:
                samples.append(result["elapsed"])
                rcodes.append(result["decoded"]["rcode_name"])
            else:
                failures += 1

        stats.append({
            "server": server,
            "queries": iterations,
            "successes": len(samples),
            "failures": failures,
            "loss_rate": round((failures / iterations) * 100, 2),
            "avg_ms": round(mean(samples) * 1000, 2) if samples else None,
            "min_ms": round(min(samples) * 1000, 2) if samples else None,
            "max_ms": round(max(samples) * 1000, 2) if samples else None,
            "rcode_mode": Counter(rcodes).most_common(1)[0][0] if rcodes else None,
        })

    stats.sort(key=lambda item: (item["loss_rate"], item["avg_ms"] if item["avg_ms"] is not None else float("inf")))
    return stats


control_domain = "www.example.com"
benchmark_results = benchmark_servers(control_domain, iterations=10)

print(f"Benchmark DNS para: {control_domain}")
print("Ranking dos servidores por desempenho:")
for position, row in enumerate(benchmark_results, start=1):
    avg_text = f"{row['avg_ms']} ms" if row["avg_ms"] is not None else "sem sucesso"
    min_text = f"{row['min_ms']} ms" if row["min_ms"] is not None else "-"
    max_text = f"{row['max_ms']} ms" if row["max_ms"] is not None else "-"
    print(
        f"{position}. {row['server']} | média={avg_text} | min={min_text} | max={max_text} | perda={row['loss_rate']}% | RCODE modal={row['rcode_mode']}"
    )


Benchmark DNS para: www.example.com
Ranking dos servidores por desempenho:
1. 9.9.9.9 | média=6.33 ms | min=4.42 ms | max=10.66 ms | perda=0.0% | RCODE modal=NOERROR
2. 1.1.1.1 | média=16.83 ms | min=14.75 ms | max=19.45 ms | perda=0.0% | RCODE modal=NOERROR
3. 8.8.8.8 | média=27.32 ms | min=21.81 ms | max=37.16 ms | perda=0.0% | RCODE modal=NOERROR
4. 208.67.222.222 | média=29.43 ms | min=25.68 ms | max=44.14 ms | perda=0.0% | RCODE modal=NOERROR


In [20]:
domain_to_scan = "www.example.com"  # altere aqui para consultar outro domínio

results, scan_rows, consensus_ip = run_multi_server_scan(domain_to_scan)

print(f"Resultado do scanner DNS para: {domain_to_scan}")
print(f"IP de consenso observado: {consensus_ip if consensus_ip else 'nenhum'}")
print()

for row in scan_rows:
    ips_text = ", ".join(row["ips"]) if row["ips"] else "sem IPs"
    notes_text = ", ".join(row["notes"]) if row["notes"] else "ok"
    print(f"Servidor {row['server']}: {row['status']} | {row['time_ms']} ms | RCODE={row['rcode']} | IPs={ips_text} | {notes_text}")


Resultado do scanner DNS para: www.example.com
IP de consenso observado: 104.20.23.154

Servidor 8.8.8.8: ok | 40.14 ms | RCODE=NOERROR | IPs=104.20.23.154, 172.66.147.243 | ok
Servidor 1.1.1.1: ok | 16.25 ms | RCODE=NOERROR | IPs=172.66.147.243, 104.20.23.154 | ok
Servidor 9.9.9.9: ok | 5.19 ms | RCODE=NOERROR | IPs=104.20.23.154, 172.66.147.243 | ok
Servidor 208.67.222.222: ok | 31.65 ms | RCODE=NOERROR | IPs=172.66.147.243, 104.20.23.154 | ok
